# NYC 311 Dataset Extraction

## Project: CivicFix

### Objective
Extract approximately 20,000 representative NYC 311 service requests for predicting whether a complaint will be resolved within 7 days.

In [1]:

import pandas as pd

# NYC 311 dataset API
url = "https://data.cityofnewyork.us/resource/erm2-nwe9.json"

# Columns required for our initial sample
columns = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "borough"
]

# Request only 5 records
query = "$select=" + ",".join(columns) + "&$limit=5"

# Fetch records
df = pd.read_json(f"{url}?{query}")

# Display the result
df


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,incident_zip,borough
0,67001434,2025-11-30T11:58:57.000,2025-12-01T20:09:35.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,10032,MANHATTAN
1,67001435,2025-11-30T13:49:17.000,2025-12-01T02:05:58.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,10032,MANHATTAN
2,67001436,2025-11-30T13:49:02.000,2025-12-01T18:13:50.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,11230,BROOKLYN
3,67001437,2025-11-30T13:48:42.000,2025-12-01T17:12:14.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,10472,BRONX
4,67001438,2025-11-30T10:37:01.000,2025-12-02T09:13:40.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,10453,BRONX


In [2]:

# Check the number of rows and columns
print("Dataset shape:", df.shape)

# Check column data types
print("\nColumn data types:")
print(df.dtypes)

# Check missing values
print("\nMissing values:")
print(df.isnull().sum())


Dataset shape: (5, 9)

Column data types:
unique_key        int64
created_date        str
closed_date         str
agency              str
complaint_type      str
descriptor          str
location_type       str
incident_zip      int64
borough             str
dtype: object

Missing values:
unique_key        0
created_date      0
closed_date       0
agency            0
complaint_type    0
descriptor        0
location_type     0
incident_zip      0
borough           0
dtype: int64


In [3]:

from urllib.parse import urlencode

year_counts = []

for year in range(2020, 2026):

    # Select complaints created during this year
    # and having a recorded closure date
    where_condition = (
        f"created_date >= '{year}-01-01T00:00:00' "
        f"AND created_date < '{year + 1}-01-01T00:00:00' "
        f"AND closed_date IS NOT NULL"
    )

    params = {
        "$select": "count(*) as total",
        "$where": where_condition
    }

    query_url = f"{url}?{urlencode(params)}"

    result = pd.read_json(query_url)

    year_counts.append({
        "year": year,
        "closed_records": int(result.loc[0, "total"])
    })

# Display yearly counts
counts_df = pd.DataFrame(year_counts)

counts_df


,year,closed_records
0,2020,2905697
1,2021,3189325
2,2022,3144776
3,2023,3162848
4,2024,3393627
5,2025,3601100


In [4]:

# Total records we want
target_sample = 20000

# Calculate proportional sample size for each year
counts_df["sample_size"] = (
    counts_df["closed_records"] /
    counts_df["closed_records"].sum() *
    target_sample
).round().astype(int)

# Correct any rounding difference
difference = target_sample - counts_df["sample_size"].sum()

counts_df.loc[
    counts_df["year"] == 2025, "sample_size"
] += difference

# Display the sampling plan
counts_df



,year,closed_records,sample_size
0,2020,2905697,2996
1,2021,3189325,3288
2,2022,3144776,3242
3,2023,3162848,3261
4,2024,3393627,3499
5,2025,3601100,3714


In [6]:
import time

time.sleep(10)

# Retry the same API request
result = pd.read_json(query_url)

display(result)

,unique_key,created_date
0,40287734,2020-09-25T15:43:34.000
1,40935119,2020-09-25T15:51:00.000
2,40962915,2020-09-25T15:51:54.000
3,41499422,2020-09-24T13:24:00.000
4,41680133,2020-09-25T16:00:27.000


In [7]:
params = {
    "$select": "unique_key, created_date",
    "$where": (
        "created_date >= '2020-01-01T00:00:00' "
        "AND created_date < '2021-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$order": "created_date ASC, unique_key ASC",
    "$limit": 10
}

query_url = f"{url}?{urlencode(params)}"

result = pd.read_json(query_url)

display(result)

,unique_key,created_date
0,45285347,2020-01-01T00:00:00.000
1,45285651,2020-01-01T00:00:00.000
2,45285821,2020-01-01T00:00:00.000
3,45287907,2020-01-01T00:00:00.000
4,45288120,2020-01-01T00:00:00.000
5,45288240,2020-01-01T00:00:00.000
6,45288728,2020-01-01T00:00:00.000
7,45289555,2020-01-01T00:00:00.000
8,45289558,2020-01-01T00:00:00.000
9,45282225,2020-01-01T00:01:12.000


In [9]:
import time
import requests

def fetch_data(params, retries=5):

    query_url = f"{url}?{urlencode(params)}"

    for attempt in range(retries):

        try:
            response = requests.get(query_url, timeout=30)
            response.raise_for_status()

            return pd.DataFrame(response.json())

        except requests.exceptions.RequestException as error:

            print(f"Attempt {attempt + 1} failed: {error}")

            if attempt < retries - 1:
                time.sleep(10 * (attempt + 1))

    raise Exception("API unavailable after multiple attempts.")

In [10]:
params = {
    "$select": "count(*) as total",
    "$where": (
        "created_date >= '2020-01-01T00:00:00' "
        "AND created_date < '2020-02-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    )
}

result = fetch_data(params)

display(result)

,total
0,193330


In [11]:
monthly_counts = []

for month in range(1, 13):

    start_date = f"2020-{month:02d}-01T00:00:00"

    if month == 12:
        end_date = "2021-01-01T00:00:00"
    else:
        end_date = f"2020-{month + 1:02d}-01T00:00:00"

    params = {
        "$select": "count(*) as total",
        "$where": (
            f"created_date >= '{start_date}' "
            f"AND created_date < '{end_date}' "
            "AND closed_date IS NOT NULL"
        )
    }

    result = fetch_data(params)

    monthly_counts.append({
        "month": month,
        "records": int(result.loc[0, "total"])
    })

    print(f"Month {month} completed")

    time.sleep(3)

monthly_df = pd.DataFrame(monthly_counts)

display(monthly_df)

Month 1 completed
Month 2 completed
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Month 3 completed
Month 4 completed
Month 5 completed
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Month 6 completed
Month 7 completed
Month 8 completed
Month 9 completed
Month 10 completed
Month 11 completed
Month 12 completed


,month,records
0,1,193330
1,2,171520
2,3,178278
3,4,158047
4,5,232209
5,6,310272
6,7,313035
7,8,337150
8,9,283240
9,10,267137


In [12]:
# Total sample allocated to 2020
year_sample = 2996

# Calculate total eligible records in 2020
total_2020 = monthly_df["records"].sum()

# Calculate proportional monthly sample sizes
monthly_df["sample_size"] = (
    monthly_df["records"] / total_2020 * year_sample
).round().astype(int)

# Correct rounding difference
difference = year_sample - monthly_df["sample_size"].sum()

monthly_df.loc[
    monthly_df["month"] == 12, "sample_size"
] += difference

display(monthly_df)

print("Total monthly sample:", monthly_df["sample_size"].sum())

,month,records,sample_size
0,1,193330,199
1,2,171520,177
2,3,178278,184
3,4,158047,163
4,5,232209,239
5,6,310272,320
6,7,313035,323
7,8,337150,348
8,9,283240,292
9,10,267137,275


Total monthly sample: 2996


In [13]:
params = {
    "$select": "date_extract_m(created_date) as month, count(*) as records",
    "$where": (
        "created_date >= '2021-01-01T00:00:00' "
        "AND created_date < '2022-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$group": "month",
    "$order": "month"
}

monthly_2021 = fetch_data(params)

display(monthly_2021)

Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)


,month,records
0,1,226972
1,2,193016
2,3,247532
3,4,239663
4,5,271945
5,6,298254
6,7,299601
7,8,278845
8,9,314248
9,10,305525


In [14]:
params = {
    "$select": "date_extract_m(created_date) as month, count(*) as records",
    "$where": (
        "created_date >= '2022-01-01T00:00:00' "
        "AND created_date < '2023-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$group": "month",
    "$order": "month"
}

monthly_2022 = fetch_data(params)

display(monthly_2022)

,month,records
0,1,277661
1,2,242738
2,3,250059
3,4,234499
4,5,274294
5,6,268378
6,7,292236
7,8,266715
8,9,269588
9,10,265207


In [15]:
params = {
    "$select": "date_extract_m(created_date) as month, count(*) as records",
    "$where": (
        "created_date >= '2023-01-01T00:00:00' "
        "AND created_date < '2024-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$group": "month",
    "$order": "month"
}

monthly_2023 = fetch_data(params)

display(monthly_2023)

Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Attempt 2 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)


,month,records
0,1,241145
1,2,221799
2,3,249354
3,4,245996
4,5,278142
5,6,262540
6,7,282219
7,8,275057
8,9,275840
9,10,295086


In [16]:
params = {
    "$select": "date_extract_m(created_date) as month, count(*) as records",
    "$where": (
        "created_date >= '2024-01-01T00:00:00' "
        "AND created_date < '2025-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$group": "month",
    "$order": "month"
}

monthly_2024 = fetch_data(params)

display(monthly_2024)

,month,records
0,1,284234
1,2,237952
2,3,262945
3,4,262617
4,5,278093
5,6,298930
6,7,292626
7,8,278394
8,9,301133
9,10,299174


In [17]:
params = {
    "$select": "date_extract_m(created_date) as month, count(*) as records",
    "$where": (
        "created_date >= '2025-01-01T00:00:00' "
        "AND created_date < '2026-01-01T00:00:00' "
        "AND closed_date IS NOT NULL"
    ),
    "$group": "month",
    "$order": "month"
}

monthly_2025 = fetch_data(params)

display(monthly_2025)

Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)


,month,records
0,1,345594
1,2,251828
2,3,277291
3,4,267661
4,5,290685
5,6,302923
6,7,311891
7,8,298659
8,9,296109
9,10,329977


In [18]:
# Combine monthly counts from all six years
monthly_data = {
    2020: monthly_df,
    2021: monthly_2021,
    2022: monthly_2022,
    2023: monthly_2023,
    2024: monthly_2024,
    2025: monthly_2025
}

monthly_all = pd.concat(
    [
        df.assign(year=year)[["year", "month", "records"]]
        for year, df in monthly_data.items()
    ],
    ignore_index=True
)

# Make sure year and month are integers
monthly_all["year"] = monthly_all["year"].astype(int)
monthly_all["month"] = monthly_all["month"].astype(int)

display(monthly_all)

,year,month,records
0,2020,1,193330
1,2020,2,171520
2,2020,3,178278
3,2020,4,158047
4,2020,5,232209
...,...,...,...
67,2025,8,298659
68,2025,9,296109
69,2025,10,329977
70,2025,11,300763


In [20]:
# Convert numeric columns into proper numeric types
monthly_all["records"] = pd.to_numeric(
    monthly_all["records"]
)

monthly_all["year"] = pd.to_numeric(
    monthly_all["year"]
)

monthly_all["month"] = pd.to_numeric(
    monthly_all["month"]
)

print(monthly_all.dtypes)

year           int64
month          int64
records        int64
year_sample    int64
dtype: object


In [21]:
import numpy as np

year_samples = counts_df.set_index("year")["sample_size"].to_dict()

monthly_all["year_sample"] = monthly_all["year"].map(year_samples)

monthly_all["exact_sample"] = (
    monthly_all["records"]
    / monthly_all.groupby("year")["records"].transform("sum")
    * monthly_all["year_sample"]
)

monthly_all["sample_size"] = np.floor(
    monthly_all["exact_sample"]
).astype(int)

for year, group in monthly_all.groupby("year"):

    remaining = int(
        group["year_sample"].iloc[0] - group["sample_size"].sum()
    )

    if remaining > 0:
        indices = (
            group["exact_sample"] - group["sample_size"]
        ).nlargest(remaining).index

        monthly_all.loc[indices, "sample_size"] += 1

display(monthly_all[["year", "month", "records", "sample_size"]])

print("Total months:", len(monthly_all))
print("Total sample:", monthly_all["sample_size"].sum())

,year,month,records,sample_size
0,2020,1,193330,199
1,2020,2,171520,177
2,2020,3,178278,184
3,2020,4,158047,163
4,2020,5,232209,239
...,...,...,...,...
67,2025,8,298659,308
68,2025,9,296109,305
69,2025,10,329977,340
70,2025,11,300763,310


Total months: 72
Total sample: 20000


In [22]:
monthly_all[
    ["year", "month", "records", "sample_size"]
].to_csv(
    "../data/processed/monthly_sampling_plan.csv",
    index=False
)

print("Sampling plan saved successfully!")

Sampling plan saved successfully!


In [23]:
# Reproducible random number generator
rng = np.random.default_rng(42)

# January 2020 sampling details
year = 2020
month = 1
total_records = 193330
required_sample = 199

# Number of records to retrieve in each block
block_size = 80
number_of_blocks = 3

# Generate random starting positions
offsets = rng.choice(
    total_records - block_size,
    size=number_of_blocks,
    replace=False
)

# Columns required for our project
columns = [
    "unique_key",
    "created_date",
    "closed_date",
    "agency",
    "complaint_type",
    "descriptor",
    "location_type",
    "incident_zip",
    "borough"
]

# Retrieve records from each random position
candidate_blocks = []

for offset in offsets:

    params = {
        "$select": ",".join(columns),
        "$where": (
            "created_date >= '2020-01-01T00:00:00' "
            "AND created_date < '2020-02-01T00:00:00' "
            "AND closed_date IS NOT NULL"
        ),
        "$order": "created_date ASC, unique_key ASC",
        "$limit": block_size,
        "$offset": int(offset)
    }

    block = fetch_data(params)

    candidate_blocks.append(block)

    print(
        f"Retrieved {len(block)} records "
        f"from offset {offset}"
    )

    time.sleep(3)

# Combine candidate blocks
candidates = pd.concat(
    candidate_blocks,
    ignore_index=True
)

# Remove duplicate records
candidates = candidates.drop_duplicates(
    subset="unique_key"
)

print("\nUnique candidate records:", len(candidates))

Retrieved 80 records from offset 126495
Retrieved 80 records from offset 17247
Retrieved 80 records from offset 149566

Unique candidate records: 240


In [24]:
# Use five randomly selected positions
block_size = 100
number_of_blocks = 5

offsets = rng.choice(
    total_records - block_size,
    size=number_of_blocks,
    replace=False
)

candidate_blocks = []

for offset in offsets:

    params = {
        "$select": ",".join(columns),
        "$where": (
            "created_date >= '2020-01-01T00:00:00' "
            "AND created_date < '2020-02-01T00:00:00' "
            "AND closed_date IS NOT NULL"
        ),
        "$order": "created_date ASC, unique_key ASC",
        "$limit": block_size,
        "$offset": int(offset)
    }

    block = fetch_data(params)

    candidate_blocks.append(block)

    print(f"Retrieved {len(block)} records from offset {offset}")

    time.sleep(3)

# Combine and remove duplicates
candidates = pd.concat(
    candidate_blocks,
    ignore_index=True
).drop_duplicates(subset="unique_key")

print("\nUnique candidates:", len(candidates))

Retrieved 100 records from offset 165903
Retrieved 100 records from offset 16607
Retrieved 100 records from offset 18197
Retrieved 100 records from offset 38929
Retrieved 100 records from offset 134751

Unique candidates: 500


In [25]:
# Randomly select 199 records from the 500 candidates
january_sample = candidates.sample(
    n=199,
    random_state=42
).reset_index(drop=True)

# Check the result
print("January sample shape:", january_sample.shape)

display(january_sample.head())

January sample shape: (199, 9)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,incident_zip,borough,location_type
0,45330366,2020-01-07T11:07:08.000,2020-02-10T00:00:00.000,DOB,Building/Use,Illegal Conversion Of Residential Building/Space,11378,QUEENS,NaN
1,45484120,2020-01-27T17:20:00.000,2020-02-04T00:00:00.000,DSNY,Request Large Bulky Item Collection,Request Large Bulky Item Collection,11361,QUEENS,Sidewalk
2,45336006,2020-01-07T11:08:00.000,2020-01-09T00:00:00.000,DSNY,Request Large Bulky Item Collection,Request Large Bulky Item Collection,11234,BROOKLYN,Sidewalk
3,45305790,2020-01-03T15:48:12.000,2020-01-03T16:04:28.000,NYPD,Noise - Vehicle,Car/Truck Horn,11366,QUEENS,Street/Sidewalk
4,45302451,2020-01-03T15:41:47.000,2020-01-13T20:38:24.000,HPD,SAFETY,SMOKE DETECTOR,11213,BROOKLYN,RESIDENTIAL BUILDING


In [26]:
# Convert dates to datetime
january_sample["created_date"] = pd.to_datetime(
    january_sample["created_date"]
)

january_sample["closed_date"] = pd.to_datetime(
    january_sample["closed_date"]
)

# Validation checks
print(
    "All records from January 2020:",
    (
        (january_sample["created_date"].dt.year == 2020) &
        (january_sample["created_date"].dt.month == 1)
    ).all()
)

print(
    "All closure dates valid:",
    (
        january_sample["closed_date"] >=
        january_sample["created_date"]
    ).all()
)

print(
    "No duplicate IDs:",
    january_sample["unique_key"].is_unique
)

print("\nDate range:")
print(january_sample["created_date"].min())
print(january_sample["created_date"].max())

All records from January 2020: True
All closure dates valid: True
No duplicate IDs: True

Date range:
2020-01-03 15:41:31
2020-01-27 17:24:00


In [27]:
# Save January 2020 sample
january_sample.to_csv(
    "../data/raw/nyc_311_2020_01.csv",
    index=False
)

print("January 2020 sample saved successfully!")

January 2020 sample saved successfully!


In [28]:
# February 2020 sampling details
year = 2020
month = 2
total_records = 171520
required_sample = 177

# Retrieve twice the required sample in each block
block_size = required_sample * 2

# Generate two random starting positions
rng = np.random.default_rng(42)

offsets = rng.choice(
    total_records - block_size,
    size=2,
    replace=False
)

candidate_blocks = []

for offset in offsets:

    params = {
        "$select": ",".join(columns),
        "$where": (
            "created_date >= '2020-02-01T00:00:00' "
            "AND created_date < '2020-03-01T00:00:00' "
            "AND closed_date IS NOT NULL"
        ),
        "$order": "created_date ASC, unique_key ASC",
        "$limit": block_size,
        "$offset": int(offset)
    }

    block = fetch_data(params)

    candidate_blocks.append(block)

    print(f"Retrieved {len(block)} records from offset {offset}")

    time.sleep(3)

# Combine candidate blocks and remove duplicates
candidates = pd.concat(
    candidate_blocks,
    ignore_index=True
).drop_duplicates(subset="unique_key")

print("\nUnique candidate records:", len(candidates))

Retrieved 354 records from offset 15276
Retrieved 354 records from offset 132474

Unique candidate records: 708


In [29]:
february_sample = candidates.sample(
    n=177,
    random_state=42
).reset_index(drop=True)

print("February sample shape:", february_sample.shape)

display(february_sample.head())

February sample shape: (177, 9)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,incident_zip,borough
0,45532395,2020-02-03T19:42:57.000,2020-03-06T13:13:12.000,TLC,Taxi Complaint,Driver Complaint - Passenger,NaN,11430,QUEENS
1,45537942,2020-02-03T20:08:27.000,2020-02-05T16:11:36.000,HPD,HEAT/HOT WATER,ENTIRE BUILDING,RESIDENTIAL BUILDING,11236,BROOKLYN
2,45534409,2020-02-03T20:23:33.000,2020-02-03T21:47:15.000,NYPD,Noise - Residential,Loud Music/Party,Residential Building/House,10473,BRONX
3,45537383,2020-02-03T19:57:44.000,2020-02-03T22:06:43.000,NYPD,Noise - Street/Sidewalk,Loud Music/Party,Street/Sidewalk,11226,BROOKLYN
4,45682943,2020-02-23T19:38:43.000,2020-02-23T21:24:55.000,NYPD,Blocked Driveway,No Access,Street/Sidewalk,10307,STATEN ISLAND


In [30]:
february_sample["created_date"] = pd.to_datetime(
    february_sample["created_date"]
)

february_sample["closed_date"] = pd.to_datetime(
    february_sample["closed_date"]
)

print(
    "All records from February 2020:",
    (
        (february_sample["created_date"].dt.year == 2020) &
        (february_sample["created_date"].dt.month == 2)
    ).all()
)

print(
    "All closure dates valid:",
    (
        february_sample["closed_date"] >=
        february_sample["created_date"]
    ).all()
)

print("No duplicate IDs:", february_sample["unique_key"].is_unique)

All records from February 2020: True
All closure dates valid: True
No duplicate IDs: True


In [31]:
february_sample.to_csv(
    "../data/raw/nyc_311_2020_02.csv",
    index=False
)

print("February 2020 sample saved successfully!")

February 2020 sample saved successfully!


In [32]:
def extract_month_sample(year, month, total_records, required_sample):

    # Number of candidate records to retrieve
    block_size = required_sample * 2

    # Random starting positions
    rng = np.random.default_rng(year * 100 + month)

    offsets = rng.choice(
        total_records - block_size,
        size=2,
        replace=False
    )

    candidate_blocks = []

    for offset in offsets:

        start_date = f"{year}-{month:02d}-01T00:00:00"

        if month == 12:
            end_date = f"{year + 1}-01-01T00:00:00"
        else:
            end_date = f"{year}-{month + 1:02d}-01T00:00:00"

        params = {
            "$select": ",".join(columns),
            "$where": (
                f"created_date >= '{start_date}' "
                f"AND created_date < '{end_date}' "
                "AND closed_date IS NOT NULL"
            ),
            "$order": "created_date ASC, unique_key ASC",
            "$limit": block_size,
            "$offset": int(offset)
        }

        block = fetch_data(params)
        candidate_blocks.append(block)

        time.sleep(3)

    # Combine and remove duplicate records
    candidates = pd.concat(
        candidate_blocks,
        ignore_index=True
    ).drop_duplicates(subset="unique_key")

    # Select the required number of records
    sample = candidates.sample(
        n=required_sample,
        random_state=year * 100 + month
    ).reset_index(drop=True)

    return sample

In [33]:
march_sample = extract_month_sample(
    year=2020,
    month=3,
    total_records=178278,
    required_sample=184
)

print("March sample shape:", march_sample.shape)

display(march_sample.head())

March sample shape: (184, 9)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,location_type,incident_zip,borough
0,45736750,2020-03-02T19:54:06.000,2020-03-03T00:19:55.000,NYPD,Blocked Driveway,No Access,Street/Sidewalk,11206,BROOKLYN
1,45834183,2020-03-15T18:36:48.000,2020-03-15T22:18:14.000,NYPD,Noise - Residential,Banging/Pounding,Residential Building/House,11105,QUEENS
2,45832781,2020-03-15T19:02:57.000,2020-03-15T22:05:47.000,NYPD,Blocked Driveway,No Access,Street/Sidewalk,10468,BRONX
3,45742326,2020-03-02T19:54:24.000,2020-03-03T00:00:00.000,DOB,General Construction/Plumbing,Building Permit - None,NaN,11385,QUEENS
4,45735143,2020-03-02T20:15:00.000,2020-03-05T00:00:00.000,DSNY,Request Large Bulky Item Collection,Request Large Bulky Item Collection,Sidewalk,10009,MANHATTAN


In [34]:
march_sample["created_date"] = pd.to_datetime(march_sample["created_date"])
march_sample["closed_date"] = pd.to_datetime(march_sample["closed_date"])

print("Correct month:", (
    (march_sample["created_date"].dt.year == 2020) &
    (march_sample["created_date"].dt.month == 3)
).all())

print("Valid closure dates:", (
    march_sample["closed_date"] >= march_sample["created_date"]
).all())

print("Unique IDs:", march_sample["unique_key"].is_unique)

Correct month: True
Valid closure dates: True
Unique IDs: True


In [35]:
march_sample.to_csv(
    "../data/raw/nyc_311_2020_03.csv",
    index=False
)

print("March 2020 sample saved!")

March 2020 sample saved!


In [36]:
import calendar
import os

for month in range(4, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2020) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2020) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2020...")

    sample = extract_month_sample(
        year=2020,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    sample["created_date"] = pd.to_datetime(sample["created_date"])
    sample["closed_date"] = pd.to_datetime(sample["closed_date"])

    valid_month = (
        (sample["created_date"].dt.year == 2020) &
        (sample["created_date"].dt.month == month)
    ).all()

    valid_dates = (
        sample["closed_date"] >= sample["created_date"]
    ).all()

    unique_ids = sample["unique_key"].is_unique

    if not (valid_month and valid_dates and unique_ids):
        raise ValueError(f"Validation failed for month {month}")

    filename = f"nyc_311_2020_{month:02d}.csv"

    sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(sample)} records")

print("\n2020 remaining months completed!")


Extracting April 2020...
Saved nyc_311_2020_04.csv: 163 records

Extracting May 2020...
Saved nyc_311_2020_05.csv: 239 records

Extracting June 2020...
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Saved nyc_311_2020_06.csv: 320 records

Extracting July 2020...
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Attempt 2 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Saved nyc_311_2020_07.csv: 323 records

Extracting August 2020...
Saved nyc_311_2020_08.csv: 348 records

Extracting September 2020...
Saved nyc_311_2020_09.csv: 292 records

Extracting October 2020...
Attempt 1 failed: 503 Server Error: Service Unavailable for url: https://data.cityofnewyork.us/resource/erm2-nwe9.json?%24select=unique_key%2Ccreated_date%2Cclosed_date%2Cagency%2Ccomplaint_type%2Cdescriptor%2Clocation_type%2Cincident_zip%2Cb

In [37]:
for month in range(1, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2021) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2021) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2021...")

    sample = extract_month_sample(
        year=2021,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    sample["created_date"] = pd.to_datetime(sample["created_date"])
    sample["closed_date"] = pd.to_datetime(sample["closed_date"])

    valid_month = (
        (sample["created_date"].dt.year == 2021) &
        (sample["created_date"].dt.month == month)
    ).all()

    valid_dates = (
        sample["closed_date"] >= sample["created_date"]
    ).all()

    unique_ids = sample["unique_key"].is_unique

    if not (valid_month and valid_dates and unique_ids):
        raise ValueError(f"Validation failed for month {month}")

    filename = f"nyc_311_2021_{month:02d}.csv"

    sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(sample)} records")

print("\n2021 extraction completed!")


Extracting January 2021...
Saved nyc_311_2021_01.csv: 234 records

Extracting February 2021...
Saved nyc_311_2021_02.csv: 199 records

Extracting March 2021...
Saved nyc_311_2021_03.csv: 255 records

Extracting April 2021...
Saved nyc_311_2021_04.csv: 247 records

Extracting May 2021...
Saved nyc_311_2021_05.csv: 280 records

Extracting June 2021...
Attempt 1 failed: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))
Attempt 2 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)


ValueError: Validation failed for month 6

In [38]:
print("June sample shape:", june_sample.shape if "june_sample" in locals() else "Variable not found")

print("Correct month:", (
    (sample["created_date"].dt.year == 2021) &
    (sample["created_date"].dt.month == 6)
).all())

print("Valid closure dates:", (
    sample["closed_date"] >= sample["created_date"]
).all())

print("Unique IDs:", sample["unique_key"].is_unique)

print("\nSample date range:")
print(sample["created_date"].min())
print(sample["created_date"].max())

June sample shape: Variable not found
Correct month: True
Valid closure dates: False
Unique IDs: True

Sample date range:
2021-06-25 17:10:18
2021-06-29 11:17:00


In [39]:
june_sample = extract_month_sample(
    year=2021,
    month=6,
    total_records=298254,
    required_sample=298
)

june_sample["created_date"] = pd.to_datetime(june_sample["created_date"])
june_sample["closed_date"] = pd.to_datetime(june_sample["closed_date"])

# Remove records with invalid closure dates
june_sample = june_sample[
    june_sample["closed_date"] >= june_sample["created_date"]
]

print("Valid June records:", len(june_sample))

Valid June records: 277


In [40]:
def extract_valid_month_sample(year, month, total_records, required_sample):

    block_size = required_sample * 2
    number_of_blocks = 4

    rng = np.random.default_rng(year * 100 + month)

    offsets = rng.choice(
        total_records - block_size,
        size=number_of_blocks,
        replace=False
    )

    candidate_blocks = []

    start_date = f"{year}-{month:02d}-01T00:00:00"

    if month == 12:
        end_date = f"{year + 1}-01-01T00:00:00"
    else:
        end_date = f"{year}-{month + 1:02d}-01T00:00:00"

    for offset in offsets:

        params = {
            "$select": ",".join(columns),
            "$where": (
                f"created_date >= '{start_date}' "
                f"AND created_date < '{end_date}' "
                "AND closed_date IS NOT NULL"
            ),
            "$order": "created_date ASC, unique_key ASC",
            "$limit": block_size,
            "$offset": int(offset)
        }

        block = fetch_data(params)
        candidate_blocks.append(block)

        print("Retrieved:", len(block))

        time.sleep(3)

    candidates = pd.concat(
        candidate_blocks,
        ignore_index=True
    ).drop_duplicates(subset="unique_key")

    candidates["created_date"] = pd.to_datetime(candidates["created_date"])
    candidates["closed_date"] = pd.to_datetime(candidates["closed_date"])

    # Keep only valid closure dates
    candidates = candidates[
        candidates["closed_date"] >= candidates["created_date"]
    ]

    print("Valid candidates:", len(candidates))

    if len(candidates) < required_sample:
        raise ValueError("Not enough valid candidates. More records are needed.")

    return candidates.sample(
        n=required_sample,
        random_state=year * 100 + month
    ).reset_index(drop=True)

In [41]:
june_sample = extract_valid_month_sample(
    year=2021,
    month=6,
    total_records=298254,
    required_sample=298
)

print("Final June sample:", june_sample.shape)

Retrieved: 596
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 596
Retrieved: 596
Retrieved: 596
Valid candidates: 2311
Final June sample: (298, 9)


In [42]:
june_sample.to_csv(
    "../data/raw/nyc_311_2021_06.csv",
    index=False
)

print("June 2021 saved successfully!")

June 2021 saved successfully!


In [43]:
for month in range(7, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2021) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2021) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2021...")

    monthly_sample = extract_valid_month_sample(
        year=2021,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    filename = f"nyc_311_2021_{month:02d}.csv"

    monthly_sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(monthly_sample)} records")

print("\nRemaining 2021 months completed!")


Extracting July 2021...
Retrieved: 618
Retrieved: 618
Retrieved: 618
Retrieved: 618
Valid candidates: 2454
Saved nyc_311_2021_07.csv: 309 records

Extracting August 2021...
Retrieved: 574
Retrieved: 574
Retrieved: 574
Retrieved: 574
Valid candidates: 2296
Saved nyc_311_2021_08.csv: 287 records

Extracting September 2021...
Retrieved: 648
Retrieved: 648
Retrieved: 648
Retrieved: 648
Valid candidates: 2592
Saved nyc_311_2021_09.csv: 324 records

Extracting October 2021...
Retrieved: 630
Retrieved: 630
Retrieved: 630
Retrieved: 630
Valid candidates: 2503
Saved nyc_311_2021_10.csv: 315 records

Extracting November 2021...
Retrieved: 552
Retrieved: 552
Retrieved: 552
Retrieved: 552
Valid candidates: 2187
Saved nyc_311_2021_11.csv: 276 records

Extracting December 2021...
Retrieved: 508
Retrieved: 508
Retrieved: 508
Retrieved: 508
Valid candidates: 2032
Saved nyc_311_2021_12.csv: 254 records

Remaining 2021 months completed!


In [44]:
for month in range(1, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2022) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2022) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2022...")

    monthly_sample = extract_valid_month_sample(
        year=2022,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    filename = f"nyc_311_2022_{month:02d}.csv"

    monthly_sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(monthly_sample)} records")

print("\n2022 extraction completed!")


Extracting January 2022...
Retrieved: 572
Retrieved: 572
Retrieved: 572
Retrieved: 572
Valid candidates: 2288
Saved nyc_311_2022_01.csv: 286 records

Extracting February 2022...
Retrieved: 500
Retrieved: 500
Retrieved: 500
Retrieved: 500
Valid candidates: 2000
Saved nyc_311_2022_02.csv: 250 records

Extracting March 2022...
Retrieved: 516
Retrieved: 516
Retrieved: 516
Retrieved: 516
Valid candidates: 2064
Saved nyc_311_2022_03.csv: 258 records

Extracting April 2022...
Retrieved: 484
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 484
Retrieved: 484
Retrieved: 484
Valid candidates: 1936
Saved nyc_311_2022_04.csv: 242 records

Extracting May 2022...
Retrieved: 566
Retrieved: 566
Retrieved: 566
Retrieved: 566
Valid candidates: 2264
Saved nyc_311_2022_05.csv: 283 records

Extracting June 2022...
Retrieved: 554
Retrieved: 554
Retrieved: 554
Retrieved: 554
Valid candidates: 2216
Saved nyc_311_2022_06.csv: 277 recor

In [45]:
for month in range(1, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2023) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2023) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2023...")

    monthly_sample = extract_valid_month_sample(
        year=2023,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    filename = f"nyc_311_2023_{month:02d}.csv"

    monthly_sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(monthly_sample)} records")

print("\n2023 extraction completed!")


Extracting January 2023...
Retrieved: 498
Retrieved: 498
Retrieved: 498
Retrieved: 498
Valid candidates: 1988
Saved nyc_311_2023_01.csv: 249 records

Extracting February 2023...
Retrieved: 458
Retrieved: 458
Retrieved: 458
Retrieved: 458
Valid candidates: 1562
Saved nyc_311_2023_02.csv: 229 records

Extracting March 2023...
Retrieved: 514
Retrieved: 514
Retrieved: 514
Retrieved: 514
Valid candidates: 2050
Saved nyc_311_2023_03.csv: 257 records

Extracting April 2023...
Retrieved: 508
Retrieved: 508
Retrieved: 508
Retrieved: 508
Valid candidates: 2032
Saved nyc_311_2023_04.csv: 254 records

Extracting May 2023...
Retrieved: 574
Retrieved: 574
Retrieved: 574
Retrieved: 574
Valid candidates: 2296
Saved nyc_311_2023_05.csv: 287 records

Extracting June 2023...
Retrieved: 542
Retrieved: 542
Retrieved: 542
Retrieved: 542
Valid candidates: 2168
Saved nyc_311_2023_06.csv: 271 records

Extracting July 2023...
Retrieved: 582
Retrieved: 582
Retrieved: 582
Retrieved: 582
Valid candidates: 2328
Sa

In [46]:
for month in range(1, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2024) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2024) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2024...")

    monthly_sample = extract_valid_month_sample(
        year=2024,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    filename = f"nyc_311_2024_{month:02d}.csv"

    monthly_sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(monthly_sample)} records")

print("\n2024 extraction completed!")


Extracting January 2024...
Retrieved: 586
Retrieved: 586
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 586
Retrieved: 586
Valid candidates: 2344
Saved nyc_311_2024_01.csv: 293 records

Extracting February 2024...
Retrieved: 490
Retrieved: 490
Retrieved: 490
Retrieved: 490
Valid candidates: 1960
Saved nyc_311_2024_02.csv: 245 records

Extracting March 2024...
Retrieved: 542
Retrieved: 542
Retrieved: 542
Retrieved: 542
Valid candidates: 2168
Saved nyc_311_2024_03.csv: 271 records

Extracting April 2024...
Retrieved: 542
Retrieved: 542
Retrieved: 542
Retrieved: 542
Valid candidates: 2168
Saved nyc_311_2024_04.csv: 271 records

Extracting May 2024...
Retrieved: 574
Retrieved: 574
Retrieved: 574
Retrieved: 574
Valid candidates: 2296
Saved nyc_311_2024_05.csv: 287 records

Extracting June 2024...
Retrieved: 616
Retrieved: 616
Retrieved: 616
Retrieved: 616
Valid candidates: 2464
Saved nyc_311_2024_06.csv: 308 recor

In [47]:
for month in range(1, 13):

    required_sample = int(
        monthly_all.loc[
            (monthly_all["year"] == 2025) &
            (monthly_all["month"] == month),
            "sample_size"
        ].iloc[0]
    )

    total_records = int(
        monthly_all.loc[
            (monthly_all["year"] == 2025) &
            (monthly_all["month"] == month),
            "records"
        ].iloc[0]
    )

    print(f"\nExtracting {calendar.month_name[month]} 2025...")

    monthly_sample = extract_valid_month_sample(
        year=2025,
        month=month,
        total_records=total_records,
        required_sample=required_sample
    )

    filename = f"nyc_311_2025_{month:02d}.csv"

    monthly_sample.to_csv(
        f"../data/raw/{filename}",
        index=False
    )

    print(f"Saved {filename}: {len(monthly_sample)} records")

print("\n2025 extraction completed!")


Extracting January 2025...
Retrieved: 714
Retrieved: 714
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 714
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 714
Valid candidates: 2856
Saved nyc_311_2025_01.csv: 357 records

Extracting February 2025...
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 520
Retrieved: 520
Retrieved: 520
Retrieved: 520
Valid candidates: 2079
Saved nyc_311_2025_02.csv: 260 records

Extracting March 2025...
Attempt 1 failed: HTTPSConnectionPool(host='data.cityofnewyork.us', port=443): Read timed out. (read timeout=30)
Retrieved: 572
Retrieved: 572
Retrieved: 572
Retrieved: 572
Valid candidates: 2288
Saved nyc_311_2025_03.csv: 286 records

Extracting April 2025...
Retrieved: 552
Retrieved: 552
Retrieved: 552
Retrieved: 552
Valid candida

In [48]:
import glob
import os

# Get all monthly CSV files
files = glob.glob("../data/raw/nyc_311_*.csv")

print("Monthly files found:", len(files))

# Read and combine all files
dataframes = [pd.read_csv(file) for file in files]

final_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("\nFinal dataset shape:", final_df.shape)

display(final_df.head())

Monthly files found: 72

Final dataset shape: (19990, 9)


,unique_key,created_date,closed_date,agency,complaint_type,descriptor,incident_zip,borough,location_type
0,45330366,2020-01-07 11:07:08,2020-02-10 00:00:00,DOB,Building/Use,Illegal Conversion Of Residential Building/Space,11378.0,QUEENS,NaN
1,45484120,2020-01-27 17:20:00,2020-02-04 00:00:00,DSNY,Request Large Bulky Item Collection,Request Large Bulky Item Collection,11361.0,QUEENS,Sidewalk
2,45336006,2020-01-07 11:08:00,2020-01-09 00:00:00,DSNY,Request Large Bulky Item Collection,Request Large Bulky Item Collection,11234.0,BROOKLYN,Sidewalk
3,45305790,2020-01-03 15:48:12,2020-01-03 16:04:28,NYPD,Noise - Vehicle,Car/Truck Horn,11366.0,QUEENS,Street/Sidewalk
4,45302451,2020-01-03 15:41:47,2020-01-13 20:38:24,HPD,SAFETY,SMOKE DETECTOR,11213.0,BROOKLYN,RESIDENTIAL BUILDING


In [49]:
# Compare actual records with planned sample sizes

actual_counts = []

for file in files:

    month_df = pd.read_csv(file)

    filename = os.path.basename(file)

    year = int(filename.split("_")[2])
    month = int(filename.split("_")[3].replace(".csv", ""))

    actual_counts.append({
        "year": year,
        "month": month,
        "actual": len(month_df)
    })

actual_df = pd.DataFrame(actual_counts)

# Compare with our original sampling plan
comparison = monthly_all.merge(
    actual_df,
    on=["year", "month"]
)

comparison["difference"] = (
    comparison["sample_size"] - comparison["actual"]
)

# Show only months with missing records
display(
    comparison[comparison["difference"] != 0][
        ["year", "month", "sample_size", "actual", "difference"]
    ]
)

print("Total planned:", comparison["sample_size"].sum())
print("Total actual:", comparison["actual"].sum())

,year,month,sample_size,actual,difference
17,2021,6,308,298,10


Total planned: 20000
Total actual: 19990


In [50]:
june_sample = extract_valid_month_sample(
    year=2021,
    month=6,
    total_records=298254,
    required_sample=308
)

print("Corrected June sample:", june_sample.shape)

Retrieved: 616
Retrieved: 616
Retrieved: 616
Retrieved: 616
Valid candidates: 2391
Corrected June sample: (308, 9)


In [51]:
june_sample.to_csv(
    "../data/raw/nyc_311_2021_06.csv",
    index=False
)

print("June 2021 file corrected and saved!")

June 2021 file corrected and saved!


In [52]:
import pandas as pd
import glob
import os

files = glob.glob("../data/raw/nyc_311_*.csv")

dataframes = [pd.read_csv(file) for file in files]

final_df = pd.concat(dataframes, ignore_index=True)

print("Monthly files:", len(files))
print("Final dataset shape:", final_df.shape)
print("Unique records:", final_df["unique_key"].nunique())

Monthly files: 72
Final dataset shape: (20000, 9)
Unique records: 20000


In [53]:
print("Missing values:")
display(final_df.isnull().sum())

print("\nDuplicate records:")
print(final_df.duplicated().sum())

print("\nDate validation:")

final_df["created_date"] = pd.to_datetime(
    final_df["created_date"], errors="coerce"
)

final_df["closed_date"] = pd.to_datetime(
    final_df["closed_date"], errors="coerce"
)

print("Invalid created dates:", final_df["created_date"].isna().sum())
print("Invalid closed dates:", final_df["closed_date"].isna().sum())

print(
    "Closed before created:",
    (final_df["closed_date"] < final_df["created_date"]).sum()
)

Missing values:


unique_key           0
created_date         0
closed_date          0
agency               0
complaint_type       0
descriptor         612
incident_zip       227
borough             27
location_type     2530
dtype: int64


Duplicate records:
0

Date validation:
Invalid created dates: 0
Invalid closed dates: 0
Closed before created: 0


In [54]:
final_df.to_csv(
    "../data/processed/nyc_311_2020_2025.csv",
    index=False
)

print("Final dataset saved successfully!")
print("Dataset shape:", final_df.shape)

Final dataset saved successfully!
Dataset shape: (20000, 9)
